#day 11 , 6 march 2026
#CNN- for images and videos processing
#data is grid and edges formeted

In [1]:

# Scenario Question
# A machine learning engineer is designing a convolutional neural network (CNN) to process grayscale images of handwritten digits
#  (28×28 pixels). She starts with a single convolutional layer defined as follows

import torch
import torch.nn as nn

# Single Conv2D layer
conv = nn.Conv2d(
    in_channels  = 1,    # grayscale input
    out_channels = 32,   # 32 filters -> 32 feature maps
    kernel_size  = 3,    # 3x3 filter
    stride       = 1,
    padding      = 1     # SAME padding: output = same size
)

# Forward pass
x   = torch.randn(1, 1, 28, 28) # (batch, C, H, W)
out = conv(x)
print(out.shape) # torch.Size([1, 32, 28, 28])

# Inspect learnable params
print('Weights:', conv.weight.shape) # (32, 1, 3, 3)
print('Bias   :', conv.bias.shape)   # (32,)
print('Total  :', 32*1*3*3 + 32)     # = 320

torch.Size([1, 32, 28, 28])
Weights: torch.Size([32, 1, 3, 3])
Bias   : torch.Size([32])
Total  : 320


In [2]:

import tensorflow as tf
from tensorflow import keras

# Single Conv2D layer
conv = keras.layers.Conv2D(
    filters     = 32,
    kernel_size = (3, 3),
    strides     = (1, 1),
    padding     = 'same',
    activation  = 'relu',
    input_shape = (28, 28, 1)
)

# Build and summarise
model = keras.Sequential([conv])
model.build(input_shape=(None, 28, 28, 1))
model.summary()
# Output shape: (None, 28, 28, 32)
# Trainable params: 320

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 28, 28, 32)     │           320 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 320 (1.25 KB)

 Trainable params: 320 (1.25 KB)

 Non-trainable params: 0 (0.00 B)

In [3]:
# Scenario: Build a CNN from scratch to classify MNIST handwritten digits (0-9). Two conv blocks + two fully connected layers.
# Import PyTorch libraries
import torch                     # Core PyTorch library
import torch.nn as nn            # Contains neural network layers (Conv, Linear, etc.)
import torch.nn.functional as F  # Contains activation functions like ReLU

# -------------------------------------------------------------
# Define the CNN Model
# -------------------------------------------------------------
class DigitCNN(nn.Module):
    """
    A Convolutional Neural Network for handwritten digit recognition (MNIST)

    Input  : 1 x 28 x 28 image (grayscale digit)
    Output : 10 values (probability scores for digits 0–9)

    Architecture Overview:
    ------------------------------------------------
    Conv → BatchNorm → ReLU → Pool
    Conv → BatchNorm → ReLU → Pool
    Flatten
    Fully Connected
    Dropout
    Fully Connected → Output
    ------------------------------------------------
    """

    def __init__(self):
        # Call parent class constructor
        super().__init__()

        # =====================================================
        # BLOCK 1
        # Convolution → BatchNorm → ReLU → MaxPooling
        # =====================================================

        # Convolution Layer
        # Input channels  = 1  (grayscale image)
        # Output channels = 32 (32 filters)
        # Kernel size     = 3x3
        # Padding = 1 keeps spatial size same
        #
        # Output size:
        # 1 x 28 x 28  →  32 x 28 x 28
        self.conv1 = nn.Conv2d(
            in_channels=1,
            out_channels=32,
            kernel_size=3,
            padding=1
        )

        # Batch Normalization
        # Normalizes the 32 feature maps to stabilize training
        self.bn1 = nn.BatchNorm2d(32)

        # Max Pooling
        # Reduces spatial size by half
        #
        # 32 x 28 x 28  →  32 x 14 x 14
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)


        # =====================================================
        # BLOCK 2
        # Another feature extraction block
        # =====================================================

        # Second Convolution Layer
        #
        # Input  = 32 channels
        # Output = 64 channels
        # Kernel = 3x3
        #
        # 32 x 14 x 14 → 64 x 14 x 14
        self.conv2 = nn.Conv2d(
            in_channels=32,
            out_channels=64,
            kernel_size=3,
            padding=1
        )

        # Batch Normalization for 64 feature maps
        self.bn2 = nn.BatchNorm2d(64)

        # Max Pooling again
        #
        # 64 x 14 x 14 → 64 x 7 x 7
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)


        # =====================================================
        # CLASSIFIER PART (Fully Connected Layers)
        # =====================================================

        # Dropout layer
        # Randomly disables 50% of neurons during training
        # Helps prevent overfitting
        self.drop = nn.Dropout(p=0.5)

        # First Fully Connected Layer
        #
        # After pooling we have:
        # 64 feature maps of size 7x7
        #
        # Flatten size = 64 * 7 * 7 = 3136
        #
        # So input features = 3136
        # Output neurons = 128
        self.fc1 = nn.Linear(
            in_features=64 * 7 * 7,
            out_features=128
        )

        # Final Fully Connected Layer
        #
        # Converts 128 features into 10 output classes
        # (digits 0 to 9)
        self.fc2 = nn.Linear(
            in_features=128,
            out_features=10
        )


    # =====================================================
    # Forward Pass
    # Defines how data flows through the network
    # =====================================================
    def forward(self, x):

        # ----- BLOCK 1 -----

        # Step 1: Convolution (feature detection)
        # Step 2: BatchNorm (stabilize values)
        # Step 3: ReLU activation (remove negative values)
        # Step 4: MaxPool (reduce spatial size)
        #
        # Input  : 1 x 28 x 28
        # Output : 32 x 14 x 14
        x = self.pool1(
                F.relu(
                    self.bn1(
                        self.conv1(x)
                    )
                )
            )


        # ----- BLOCK 2 -----

        # Input  : 32 x 14 x 14
        # Output : 64 x 7 x 7
        x = self.pool2(
                F.relu(
                    self.bn2(
                        self.conv2(x)
                    )
                )
            )


        # ----- FLATTEN -----

        # Convert feature maps into a vector
        #
        # 64 x 7 x 7 → 3136
        #
        # x.size(0) keeps batch size unchanged
        x = x.view(x.size(0), -1)


        # ----- FULLY CONNECTED LAYER -----

        # Linear layer
        # Apply ReLU
        # Apply Dropout
        #
        # 3136 → 128
        x = self.drop(
                F.relu(
                    self.fc1(x)
                )
            )


        # ----- OUTPUT LAYER -----

        # 128 → 10
        #
        # These are called "logits"
        # Softmax is usually applied later in the loss function
        return self.fc2(x)


# -------------------------------------------------------------
# Create the Model
# -------------------------------------------------------------
model = DigitCNN()

# Print model architecture
print(model)

DigitCNN(
  (conv1): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (pool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (pool2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (drop): Dropout(p=0.5, inplace=False)
  (fc1): Linear(in_features=3136, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=10, bias=True)
)


In [ ]:
#  Scenario: Handwritten Digit Recognition System
# A data science team is tasked with building an AI model that can automatically
#  recognize handwritten digits (0–9) from scanned forms. They decide to use the
#   MNIST dataset, which contains thousands of grayscale images of digits written
#   by different people

# Import optimizer module from PyTorch
import torch.optim as optim

# Import MNIST dataset and image transformations
from torchvision import datasets, transforms

# DataLoader helps load data in batches during training
from torch.utils.data import DataLoader


# -----------------------------
# DATA PREPARATION
# -----------------------------

# Compose multiple image transformations together
transform = transforms.Compose([

    # Convert image (PIL format) into PyTorch Tensor
    transforms.ToTensor(),

    # Normalize pixel values
    # MNIST mean = 0.1307
    # MNIST std  = 0.3081
    # This helps the model train faster and more stable
    transforms.Normalize((0.1307,), (0.3081,))
])


# Download and load the MNIST training dataset
train_ds = datasets.MNIST(
    '.',                # dataset saved in current folder
    train=True,         # load training set
    download=True,      # download if not already present
    transform=transform # apply the transformations above
)

# Download and load the MNIST test dataset
test_ds = datasets.MNIST(
    '.',
    train=False,        # load test set
    download=True,
    transform=transform
)


# DataLoader loads dataset in batches for training
train_dl = DataLoader(
    train_ds,
    batch_size=64,      # process 64 images at a time
    shuffle=True        # shuffle data every epoch
)

# DataLoader for test data
test_dl = DataLoader(
    test_ds,
    batch_size=64       # no need to shuffle test data
)


# -----------------------------
# MODEL, LOSS, OPTIMIZER
# -----------------------------

# Create instance of the CNN model defined earlier
model = DigitCNN()


# Loss function used for classification
# CrossEntropyLoss automatically applies Softmax
criterion = nn.CrossEntropyLoss()


# Optimizer updates model weights using gradients
optimizer = optim.Adam(

    model.parameters(), # all trainable parameters in the model

    lr = 1e-3           # learning rate = 0.001
)


# -----------------------------
# TRAINING LOOP
# -----------------------------

# Train the model for 10 epochs
# (1 epoch = model sees entire dataset once)
for epoch in range(10):

    # Set model to training mode
    model.train()

    # Loop through batches of training data
    for imgs, labels in train_dl:

        # Reset gradients from previous step
        optimizer.zero_grad()

        # Forward pass
        # Pass images through the CNN
        predictions = model(imgs)

        # Compute loss between predictions and actual labels
        loss = criterion(predictions, labels)

        # Backpropagation
        # Calculate gradients of loss w.r.t model parameters
        loss.backward()

        # Update model weights using optimizer
        optimizer.step()


# -----------------------------
# MODEL EVALUATION
# -----------------------------

# Switch model to evaluation mode
model.eval()

# Counter for correct predictions
correct = 0


# Disable gradient computation
# (saves memory and speeds up evaluation)
with torch.no_grad():

    # Loop through test dataset
    for imgs, labels in test_dl:

        # Get model predictions
        outputs = model(imgs)

        # Get predicted class (index of highest score)
        predicted = outputs.argmax(1)

        # Count correct predictions
        correct += (predicted == labels).sum().item()


# -----------------------------
# ACCURACY CALCULATION
# -----------------------------

print(f'Accuracy: {correct/len(test_ds)*100:.2f}%')

100%|██████████| 9.91M/9.91M [00:00<00:00, 22.5MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 602kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 5.63MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 11.4MB/s]


In [ ]:
# Scenario Question

# A biomedical researcher is designing a convolutional neural network (CNN) to analyze microscopic grayscale images of blood cells (64×64 pixels). She begins with a single convolutional layer defined as follows:

# Input: 64×64 grayscale images (1 channel)
# Convolutional Layer: 16 filters, each of size 5×5
# Stride: 1
# Padding: Same
# Activation Function: ReLU
# The goal of this layer is to extract local structural features such as cell boundaries and texture variations,
#  which will later help in classifying whether the blood cells are healthy or show signs of abnormality.
# Import required libraries
import tensorflow as tf
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, Input

# Step 1: Define the CNN model
model = Sequential()

# Input layer (64x64 grayscale image)
model.add(Input(shape=(64, 64, 1)))

# Convolution layer
model.add(Conv2D(
    filters=16,          # number of filters
    kernel_size=(5,5),   # filter size
    strides=1,           # stride
    padding='same',      # same padding
    activation='relu'    # activation function
))

# Step 2: Display model architecture
model.summary()

# Step 3: Create a sample grayscale image (dummy data)
sample_image = np.random.rand(1, 64, 64, 1)

# Step 4: Pass image through CNN
output = model.predict(sample_image)

# Step 5: Print output shape
print("Input Shape:", sample_image.shape)
print("Output Shape:", output.shape)

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_1 (Conv2D)               │ (None, 64, 64, 16)     │           416 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 416 (1.62 KB)

 Trainable params: 416 (1.62 KB)

 Non-trainable params: 0 (0.00 B)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 240ms/step
Input Shape: (1, 64, 64, 1)
Output Shape: (1, 64, 64, 16)
